# 07 — NLP Token Name Risk Classifier

Klasifikasi token scam/clean berdasarkan symbol + name.
Model: DistilBERT fine-tuned (binary classification).

⚠️ Fine-tune butuh GPU (atau patient CPU + small batch). Inference cepat di CPU.

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

In [ ]:
# Synthetic labeled dataset (ganti dengan CryptoScamDB + legitimate token list)
np.random.seed(42)
n = 500
scam_names = ['PEPE2', 'ELONMOON', 'SHIBAROCKET', 'BABYDOGE2', 'FLOKI100X',
              'SAFEMOON2', '100XCOIN', 'ROCKETMOON', 'SHIBELON', 'DOGECOIN2']
clean_names = ['UNISWAP', 'AAVE', 'COMP', 'MAKER', 'LINK', 'UNI', 'CRV', 'BAL', 'SNX', 'YFI']

tokens = []
for _ in range(n//2):
    base = np.random.choice(scam_names)
    tokens.append({'name': base, 'symbol': base[:8], 'label': 1})
for _ in range(n//2):
    base = np.random.choice(clean_names)
    tokens.append({'name': base, 'symbol': base[:5], 'label': 0})

df = pd.DataFrame(tokens)
print(f'Total: {len(df)}, scam: {df.label.sum()}, clean: {(df.label==0).sum()}')

## Feature Engineering — Character-level features (sebelum transformer)

In [ ]:
import re
def extract_char_features(df):
    features = pd.DataFrame()
    features['name_len'] = df['name'].str.len()
    features['symbol_len'] = df['symbol'].str.len()
    features['name_upper_ratio'] = df['name'].apply(lambda s: sum(1 for c in s if c.isupper())/max(len(s),1))
    features['symbol_has_number'] = df['symbol'].str.contains(r'\d').astype(int)
    features['name_has_x'] = df['name'].str.lower().str.contains('x').astype(int)
    features['name_has_moon'] = df['name'].str.lower().str.contains('moon').astype(int)
    features['name_has_doge'] = df['name'].str.lower().str.contains('doge|shib|floki|pepe|elon').astype(int)
    features['symbol_repeat'] = df['symbol'].apply(lambda s: sum(1 for i in range(len(s)-1) if s[i]==s[i+1]))
    return features

X_char = extract_char_features(df)
X_char.head(3)

In [ ]:
# Baseline: simple classifier using char features
from sklearn.ensemble import RandomForestClassifier
X_train, X_test, y_train, y_test = train_test_split(X_char, df['label'], test_size=0.2, random_state=42)
rf = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_train, y_train)
print(classification_report(y_test, rf.predict(X_test), target_names=['clean', 'scam']))

## DistilBERT Fine-tuning (uncomment if transformers installed)

```python
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, Trainer, TrainingArguments
import torch

model_name = 'distilbert-base-uncased'
tokenizer = DistilBertTokenizer.from_pretrained(model_name)
model = DistilBertForSequenceClassification.from_pretrained(model_name, num_labels=2)

def tokenize(batch):
    return tokenizer(batch['name'], padding=True, truncation=True, max_length=32)

# ... fine-tune with Trainer API ...
```

⚠️ Uncomment dan jalankan hanya jika GPU tersedia atau pakai Google Colab.